# AiXbio — Notebook 5: pSSR Attack + Feature Taxonomy

**pSSR** (Protein Subspace Rerouting): gradient attack on probe embeddings.
**Feature Taxonomy**: identify biological meaning of top SAE features.

Runs on pre-computed embeddings — no ESM-2 reload needed.

In [1]:
import warnings, json
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from torch.utils.data import TensorDataset, DataLoader
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
BEST_LAYER = 33
Path('results/pssr').mkdir(parents=True, exist_ok=True)
print(f'Device: {DEVICE}')

# Load embeddings
tox_embs  = np.load(f'embeddings/natural_toxins_layer{BEST_LAYER}.npy')
ctrl_embs = np.load(f'embeddings/controls_layer{BEST_LAYER}.npy')
rdsg_embs = np.load(f'embeddings/redesigns_layer{BEST_LAYER}.npy')
print(f'Tox: {tox_embs.shape}  Ctrl: {ctrl_embs.shape}  Rdsg: {rdsg_embs.shape}')

# Retrain probe
class ToxinProbe(nn.Module):
    def __init__(self, d=1280):
        super().__init__()
        self.linear = nn.Linear(d, 1)
    def forward(self, x):
        return self.linear(x).squeeze(-1)

X = np.concatenate([tox_embs, ctrl_embs])
y = np.concatenate([np.ones(len(tox_embs)), np.zeros(len(ctrl_embs))])
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

probe = ToxinProbe().to(DEVICE)
ds    = TensorDataset(torch.tensor(X_tr, dtype=torch.float32),
                      torch.tensor(y_tr,  dtype=torch.float32))
crit  = nn.BCEWithLogitsLoss()
opt   = torch.optim.Adam(probe.parameters(), lr=1e-2, weight_decay=1e-4)
probe.train()
for _ in range(150):
    for xb, yb in DataLoader(ds, batch_size=256, shuffle=True):
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        opt.zero_grad(); crit(probe(xb), yb).backward(); opt.step()
probe.eval()

with torch.no_grad():
    te_sc = torch.sigmoid(probe(torch.tensor(X_te, dtype=torch.float32).to(DEVICE))).cpu().numpy()
print(f'Probe AUROC: {roc_auc_score(y_te, te_sc):.4f}')

probe_dir = probe.linear.weight.data[0].cpu().numpy()  # (1280,)

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


Device: cuda
Tox: (1712, 1280)  Ctrl: (2072, 1280)  Rdsg: (643, 1280)
Probe AUROC: 0.9971


## pSSR Attack

Gradient attack on toxin embeddings: minimise probe confidence within L2 ball.
An oracle attack — requires full white-box access to probe weights.

In [8]:
def pssr_attack(embeddings, epsilon=2.0, n_steps=500, lr=0.01):
    """
    Protein Subspace Rerouting: gradient descent minimising probe confidence.
    Constrained to L2 ball of radius epsilon around original embedding.
    epsilon=2.0 ~ 1 std of natural embedding variation.
    """
    e_orig = torch.tensor(embeddings, dtype=torch.float32).to(DEVICE)
    delta  = torch.zeros_like(e_orig, requires_grad=True)
    opt    = torch.optim.Adam([delta], lr=lr)

    for _ in range(n_steps):
        
        e_adv = e_orig + delta
        loss  = torch.sigmoid(probe(e_adv)).mean()   # want to MINIMISE this
        loss.backward()       # gradient descent → reduces toxin score
        opt.step()
        opt.zero_grad()
        with torch.no_grad():
            norms = delta.norm(dim=-1, keepdim=True).clamp(min=1e-8)
            delta.data = delta.data * (norms.clamp(max=epsilon) / norms)


        e_adv = (e_orig + delta).detach()
    
    with torch.no_grad():
        adv_scores = torch.sigmoid(probe(e_adv)).cpu().numpy()
        orig_scores = torch.sigmoid(probe(e_orig)).cpu().numpy()
    
    evasion = float((adv_scores < 0.3).mean())
    delta_np = delta.detach().cpu().numpy()
    d_mean = delta_np.mean(0)
    d_norm = d_mean / (np.linalg.norm(d_mean) + 1e-8)
    p_norm = probe_dir / (np.linalg.norm(probe_dir) + 1e-8)
    cos = float(np.dot(d_norm, p_norm))

    print(f'  ε={epsilon}: orig_score={orig_scores.mean():.3f} → adv_score={adv_scores.mean():.3f}  '
          f'evasion={evasion:.1%}  cosine={cos:+.3f}')

    return {'evasion_rate': evasion, 'cosine_with_probe': cos,
            'adv_scores_mean': float(adv_scores.mean()),
            'orig_scores_mean': float(orig_scores.mean()),
            'epsilon': epsilon, 'delta': delta_np}


# Baseline: random redesign evasion
with torch.no_grad():
    rdsg_scores = torch.sigmoid(probe(torch.tensor(rdsg_embs, dtype=torch.float32).to(DEVICE))).cpu().numpy()
baseline_evasion = float((rdsg_scores < 0.3).mean())
print(f'Baseline (random ProteinMPNN) evasion rate: {baseline_evasion:.1%}')

# pSSR sweep over epsilon
print('\nRunning pSSR attack on first 100 toxin embeddings...')
pssr_results = {}
for eps in [0.5, 1.0, 2.0, 5.0, 10.0]:
    r = pssr_attack(tox_embs[:100], epsilon=eps)
    pssr_results[eps] = r
    print(f'  epsilon={eps:5.1f}: evasion={r["evasion_rate"]:.1%}  '
          f'cosine(delta, probe)={r["cosine_with_probe"]:+.3f}  '
          f'mean_score={r["adv_scores_mean"]:.3f}')

Baseline (random ProteinMPNN) evasion rate: 8.2%

Running pSSR attack on first 100 toxin embeddings...
  ε=0.5: orig_score=0.977 → adv_score=0.153  evasion=81.0%  cosine=-0.805
  epsilon=  0.5: evasion=81.0%  cosine(delta, probe)=-0.805  mean_score=0.153
  ε=1.0: orig_score=0.977 → adv_score=0.000  evasion=100.0%  cosine=-0.805
  epsilon=  1.0: evasion=100.0%  cosine(delta, probe)=-0.805  mean_score=0.000
  ε=2.0: orig_score=0.977 → adv_score=0.000  evasion=100.0%  cosine=-0.805
  epsilon=  2.0: evasion=100.0%  cosine(delta, probe)=-0.805  mean_score=0.000
  ε=5.0: orig_score=0.977 → adv_score=0.000  evasion=100.0%  cosine=-0.806
  epsilon=  5.0: evasion=100.0%  cosine(delta, probe)=-0.806  mean_score=0.000
  ε=10.0: orig_score=0.977 → adv_score=0.000  evasion=100.0%  cosine=-0.806
  epsilon= 10.0: evasion=100.0%  cosine(delta, probe)=-0.806  mean_score=0.000


## Evasion Comparison Table

In [9]:
print('='*62)
print(f'{"Attack":<38} {"Evasion Rate":>12}  Notes')
print('='*62)
print(f'{"BLAST @ 30% threshold":<38} {0.0:>12.1%}')
print(f'{"BLAST @ 40% threshold":<38} {0.0:>12.1%}')
print(f'{"Random ProteinMPNN (seq-space)":<38} {baseline_evasion:>12.1%}  no model access')
for eps in [2.0, 5.0, 10.0]:
    r = pssr_results[eps]
    print(f'{f"pSSR ε={eps} (embedding-space)":<38} {r["evasion_rate"]:>12.1%}  white-box oracle')
print('='*62)
print(f'\nKey: cosine(pSSR_delta, probe_direction)')
for eps in [2.0, 5.0]:
    r = pssr_results[eps]
    print(f'  ε={eps}: cosine = {r["cosine_with_probe"]:+.3f}',
          '← attack counters probe' if r['cosine_with_probe'] < -0.5 else '← partially orthogonal')

Attack                                 Evasion Rate  Notes
BLAST @ 30% threshold                          0.0%
BLAST @ 40% threshold                          0.0%
Random ProteinMPNN (seq-space)                 8.2%  no model access
pSSR ε=2.0 (embedding-space)                 100.0%  white-box oracle
pSSR ε=5.0 (embedding-space)                 100.0%  white-box oracle
pSSR ε=10.0 (embedding-space)                100.0%  white-box oracle

Key: cosine(pSSR_delta, probe_direction)
  ε=2.0: cosine = -0.805 ← attack counters probe
  ε=5.0: cosine = -0.806 ← attack counters probe


## Feature Taxonomy — What Do Top SAE Features Encode?

For top 5 structure-robust features (transfer > 0.8), find:
- Which sequences activate them most strongly
- What amino acid patterns they prefer
- Biological interpretation

In [10]:
from Bio import SeqIO
import collections

# Load SAE activations (from Notebook 3)
try:
    tox_acts  = np.load('embeddings/sae_activations_toxins.npy')
    rdsg_acts = np.load('embeddings/sae_activations_redesigns.npy')
except FileNotFoundError:
    print('SAE activations not pre-saved. Loading SAE to recompute...')
    from interplm.sae.inference import load_sae_from_hf
    sae = load_sae_from_hf(plm_model='esm2-650m', plm_layer=BEST_LAYER)
    sae = sae.to(DEVICE).eval()
    def get_acts(embs, bs=256):
        out = []
        with torch.no_grad():
            for i in range(0, len(embs), bs):
                b = torch.tensor(embs[i:i+bs], dtype=torch.float32).to(DEVICE)
                try:    out.append(sae.encode(b).cpu().numpy())
                except: out.append(sae(b)[1].cpu().numpy())
        return np.concatenate(out)
    tox_acts  = get_acts(tox_embs)
    rdsg_acts = get_acts(rdsg_embs)
    np.save('embeddings/sae_activations_toxins.npy',   tox_acts)
    np.save('embeddings/sae_activations_redesigns.npy', rdsg_acts)
    print(f'SAE activations: tox={tox_acts.shape}  rdsg={rdsg_acts.shape}')

# Load transfer results from main_results
main = json.load(open('results/main_results.json'))
transfer_data = main['sae']['feature_transfer']

# Robust features (transfer > 0.8)
robust_feats = [d['feature'] for d in transfer_data if d['transfer_ratio'] > 0.8]
print(f'Structure-robust features: {robust_feats}')

# Load toxin sequences for motif analysis
tox_seqs = {r.id: str(r.seq) for r in
            SeqIO.parse('data/toxins_clustered.fasta', 'fasta')}
seq_ids  = list(tox_seqs.keys())
seqs_arr = list(tox_seqs.values())

Structure-robust features: [6122, 4097, 1055, 9927, 6971, 9487, 5406, 8112, 9242]


In [11]:
# Amino acid frequency analysis per robust feature
def aa_profile(feature_idx, top_n=20):
    """Find top-activating sequences and their AA composition."""
    acts = tox_acts[:, feature_idx]
    top_idx = np.argsort(acts)[::-1][:top_n]

    # AA composition in top sequences
    aa_counts = collections.Counter()
    for i in top_idx:
        if i < len(seqs_arr):
            aa_counts.update(seqs_arr[i])
    total = sum(aa_counts.values())
    aa_freq = {aa: cnt/total for aa, cnt in aa_counts.most_common()}

    # Reference AA freq across all toxins
    ref_counts = collections.Counter()
    for s in seqs_arr: ref_counts.update(s)
    ref_total = sum(ref_counts.values())
    ref_freq  = {aa: cnt/ref_total for aa, cnt in ref_counts.items()}

    # Enrichment = top_freq / ref_freq
    enrichment = {aa: aa_freq.get(aa, 0) / (ref_freq.get(aa, 1e-4))
                  for aa in 'ACDEFGHIKLMNPQRSTVWY'}
    top_enriched = sorted(enrichment, key=enrichment.get, reverse=True)[:5]

    return {
        'feature': feature_idx,
        'top_sequences': [seq_ids[i] for i in top_idx if i < len(seq_ids)],
        'top_enriched_aa': top_enriched,
        'enrichment': {aa: enrichment[aa] for aa in top_enriched},
        'mean_activation': float(acts[top_idx].mean()),
        'activation_rate': float((acts > 0).mean()),
    }


print('Feature Taxonomy — Top Enriched Amino Acids:')
print(f'{"Feature":>8}  {"Transfer":>9}  {"Top AAs (enriched)"}  {"Interpretation"}')
print('-'*75)

taxonomy = {}
for d in transfer_data:
    f = d['feature']
    if d['transfer_ratio'] < 0.8 and d['transfer_ratio'] > 0.3:
        continue  # skip intermediate
    profile = aa_profile(f)
    taxonomy[f] = profile
    top_aa = ', '.join([f"{aa}({profile['enrichment'][aa]:.1f}x)"
                        for aa in profile['top_enriched_aa'][:3]])
    t_class = 'ROBUST'   if d['transfer_ratio'] > 0.8 else 'EVADABLE'
    # Biological guess based on AA composition
    bio_guess = ''
    aas = profile['top_enriched_aa']
    if 'C' in aas[:2]: bio_guess = 'Disulfide/cysteine-rich'
    elif 'G' in aas[:2] or 'P' in aas[:2]: bio_guess = 'Structural rigidity'
    elif 'K' in aas[:2] or 'R' in aas[:2]: bio_guess = 'Charged/ionic surface'
    elif 'H' in aas[:2]: bio_guess = 'Histidine coordination'
    else: bio_guess = f'Enriched in {aas[0]}'
    print(f'{f:>8}  {d["transfer_ratio"]:>9.2f}  {top_aa:<35}  {bio_guess}  [{t_class}]')

Feature Taxonomy — Top Enriched Amino Acids:
 Feature   Transfer  Top AAs (enriched)  Interpretation
---------------------------------------------------------------------------
    6122       2.41  H(1.6x), F(1.5x), P(1.3x)            Histidine coordination  [ROBUST]
    4097       2.64  P(2.6x), K(1.5x), E(1.3x)            Structural rigidity  [ROBUST]
    5312       0.13  C(4.7x), W(2.5x), G(2.1x)            Disulfide/cysteine-rich  [EVADABLE]
    1055       3.36  E(2.3x), A(1.2x), Q(1.2x)            Enriched in E  [ROBUST]
    9026       0.07  C(3.3x), H(1.2x), P(1.2x)            Disulfide/cysteine-rich  [EVADABLE]
    4397       0.00  W(1.3x), H(1.3x), P(1.2x)            Histidine coordination  [EVADABLE]
    9927       0.96  C(3.6x), G(1.9x), W(1.9x)            Disulfide/cysteine-rich  [ROBUST]
    6971       2.19  P(1.2x), V(1.2x), D(1.1x)            Structural rigidity  [ROBUST]
    2704       0.00  I(1.2x), H(1.2x), A(1.1x)            Histidine coordination  [EVADABLE]
    1974

## Save All pSSR + Taxonomy Results

In [12]:
def _conv(o):
    if isinstance(o, (np.integer, np.floating)): return float(o)
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, dict): return {str(k): _conv(v) for k, v in o.items()}
    return o

pssr_summary = {
    'baseline_evasion': baseline_evasion,
    'results_by_epsilon': {
        str(eps): {
            'evasion_rate':      r['evasion_rate'],
            'cosine_with_probe': r['cosine_with_probe'],
            'adv_scores_mean':   r['adv_scores_mean'],
        } for eps, r in pssr_results.items()
    },
    'finding': (
        f'Random ProteinMPNN evasion: {baseline_evasion:.1%}. '
        f'pSSR oracle evasion at ε=5: {pssr_results[5.0]["evasion_rate"]:.1%}. '
        f'cosine(attack, probe) = {pssr_results[2.0]["cosine_with_probe"]:+.3f}.'
    )
}

pssr_summary['taxonomy'] = {str(k): _conv(v) for k, v in taxonomy.items()}

with open('results/pssr/pssr_results.json', 'w') as f:
    json.dump(pssr_summary, f, default=_conv, indent=2)

main = json.load(open('results/main_results.json'))
main['pssr'] = pssr_summary
with open('results/main_results.json', 'w') as f:
    json.dump(main, f, default=_conv, indent=2)

print('=== pSSR + Taxonomy Summary ===')
print(f'  Baseline (random redesign) evasion: {baseline_evasion:.1%}')
for eps in [2.0, 5.0, 10.0]:
    r = pssr_results[eps]
    print(f'  pSSR ε={eps}: evasion={r["evasion_rate"]:.1%}  '
          f'cosine={r["cosine_with_probe"]:+.3f}')
print(f'  Taxonomy computed for {len(taxonomy)} features')
print('Saved → results/pssr/ + main_results.json')

=== pSSR + Taxonomy Summary ===
  Baseline (random redesign) evasion: 8.2%
  pSSR ε=2.0: evasion=100.0%  cosine=-0.805
  pSSR ε=5.0: evasion=100.0%  cosine=-0.806
  pSSR ε=10.0: evasion=100.0%  cosine=-0.806
  Taxonomy computed for 17 features
Saved → results/pssr/ + main_results.json
